# PARC2026 — 69b OpenVLA RLDS bridge smoke (recovery)

D10で選択した `V2_SQRT_BALANCED_RAW` の8 episodeだけを変換し、容量と入力形式を確認します。**下のコードセル1つを実行するだけです。** 依存インストール・変換・レポート確認・容量判定を順番に実行し、失敗した場合はその場で停止します。

既存のD10結果、Drive dataset、選択manifest、成功済みRLDS成果物は削除しません。検証済み環境と一致するsmokeレポートは再利用します。変換をやり直す場合も専用の新しいローカル作業ディレクトリを使います。

- CPU / L4 / A100対応。GPU学習・モデル重みのロード・全episode変換は行いません。
- Python 3.10の隔離環境を使用し、PyAV 12.3.0を含む依存はbinary wheelのみ許可します。
- 失敗時は実際の子プロセスのエラー末尾とログ保存先を表示します。別のセルへ進む必要はありません。
- 完了後は `bridge_smoke_status.json` がPASSになり、変換レポートと容量判定を表示します。
- 8 episodeのsmokeでは `conversion_contract.json` は作りません。35GiBの閾値は次方式を選ぶための目安で、full変換やM3学習の開始許可ではありません。

**操作:** ランタイムを接続 → 下のセルを1回実行 → `=== 69b COMPLETE ===` を確認。以前の2/5・3/5・4/5セルや手動修正コマンドを追加実行する必要はありません。


In [ ]:
# One-click sequential recovery. No model training or full RLDS conversion.
import json, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/parc2026')
ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / 'py_AI_69b_recovery'  # Separate from the user's existing checkout.
PIN = '5f7cbb4ed4b054b381a3a0074ab54b419b09a5de'
URL = 'https://github.com/yu37330/py_AI.git'
if not (REPO / '.git').exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f'Existing non-Git directory: {REPO}. Please inspect it before continuing.')
    subprocess.run(['git', 'clone', '--no-checkout', URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', PIN], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', PIN], check=True)
got = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if got != PIN:
    raise RuntimeError(f'Repository pin mismatch: {got}')
print('69b recovery code:', got, flush=True)

cmd = [sys.executable, '-u', str(REPO / 'tools/colab/rlds_smoke_recovery.py'),
       '--root', str(ROOT), '--repo', str(REPO),
       '--drive', '/content/drive/MyDrive/parc2026-cache']
subprocess.run(cmd, check=True)

OUT = Path('/content/drive/MyDrive/parc2026-cache/openvla-rlds-selected-v1')
status = json.loads((OUT / 'bridge_smoke_status.json').read_text())
if status.get('status') != 'PASS':
    raise RuntimeError(f'69b did not complete: {status}')
print('=== 69b COMPLETE ===', flush=True)
print('Next: share the capacity decision; do not start notebook 70/M3 yet.', flush=True)
